# LeetCode 517 — Super Washing Machines bằng Python

- Link: https://leetcode.com/problems/super-washing-machines/
- Độ khó: **Hard**
- Chủ đề: Greedy, Prefix Sum (tổng tiền tố), Luồng qua "lát cắt", Chặn dưới / chặn trên

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMinMoves(self, machines: List[int]) -> int:

```

## 1. Mục tiêu học

1. Hiểu luật chơi "mỗi lượt, **mọi** máy được chọn chuyển **đồng thời** 1 bộ đồ sang máy kề" và tại sao nó khác với "mỗi lượt chuyển 1 bộ đồ".
2. Dùng **tổng tiền tố** để tính **luồng bắt buộc** đi qua mỗi ranh giới giữa hai máy.
3. Tìm **chặn dưới** (lower bound) cho số lượt từ hai góc nhìn: ranh giới (cạnh) và từng máy (đỉnh) — rồi thấy chặn dưới chính là đáp án.
4. Viết **BFS vét cạn** trên không gian trạng thái để kiểm chứng công thức trên input nhỏ.
5. So sánh 3 cách: BFS, công thức tiền tố, công thức "luồng trái/phải" theo từng máy.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `n` máy giặt xếp thành **một hàng**. Máy `i` đang chứa `machines[i]` bộ quần áo. Mỗi **lượt**, bạn chọn **bất kỳ** `m` máy (`1 <= m <= n`); **mỗi** máy được chọn chuyển **đúng một** bộ quần áo sang **một trong hai máy kề** của nó — tất cả diễn ra **cùng lúc**.

Tìm **số lượt ít nhất** để mọi máy có **cùng số** bộ quần áo. Nếu không thể, trả về `-1`.

**Ví dụ 1**

```text
Input:  machines = [1,0,5]
Output: 3
Lượt 1:  1   0 <-- 5   =>  1  1  4
Lượt 2:  1 <-- 1 <-- 4 =>  2  1  3
Lượt 3:  2   1 <-- 3   =>  2  2  2
```

**Ví dụ 2**

```text
Input:  machines = [0,3,0]
Output: 2
Lượt 1:  0 <-- 3   0   =>  1  2  0
Lượt 2:  1   2 --> 0   =>  1  1  1
```

**Ví dụ 3**

```text
Input:  machines = [0,2,0]
Output: -1
Giải thích: tổng 2 không chia hết cho 3 máy.
```

**Ràng buộc**

- `n == machines.length`, `1 <= n <= 10^4`
- `0 <= machines[i] <= 10^5`

Chú ý: ở ví dụ 2, máy giữa có 3 bộ nhưng **mỗi lượt chỉ cho đi được 1 bộ** (dù có hai hàng xóm) — nên cần 2 lượt để cho đi 2 bộ.

In [1]:
machines = [1, 0, 5]
total, n = sum(machines), len(machines)
print("tổng =", total, "| n =", n, "| chia hết?", total % n == 0, "| trung bình =", total // n)
print("mỗi máy thừa/thiếu:", [x - total // n for x in machines])

tổng = 6 | n = 3 | chia hết? True | trung bình = 2
mỗi máy thừa/thiếu: [-1, -2, 3]


## 3. Kiến thức nền

### 3.1. Phép chia hết `%` và trung bình

Mọi máy cuối cùng có cùng số `avg`, nên `n · avg = tổng` → **tổng phải chia hết cho `n`**, nếu không trả `-1`. Khi chia hết, `avg = tổng // n`.

In [2]:
for arr in [[1, 0, 5], [0, 3, 0], [0, 2, 0]]:
    s = sum(arr)
    print(arr, "tổng", s, "->", "avg = " + str(s // len(arr)) if s % len(arr) == 0 else "không chia hết -> -1")

[1, 0, 5] tổng 6 -> avg = 2
[0, 3, 0] tổng 3 -> avg = 1
[0, 2, 0] tổng 2 -> không chia hết -> -1


### 3.2. Tổng tiền tố (prefix sum) của "độ lệch"

Đặt `d[i] = machines[i] - avg` (dương: thừa, âm: thiếu). Tổng tiền tố `P[i] = d[0] + … + d[i]` cho biết **phần bên trái (gồm máy `i`) thừa bao nhiêu** so với mục tiêu. Tổng toàn bộ `P[n-1] = 0`.

In [3]:
from itertools import accumulate
machines = [1, 0, 5]
avg = sum(machines) // len(machines)
d = [x - avg for x in machines]
P = list(accumulate(d))
print("d =", d)
print("P =", P)

d = [-1, -2, 3]
P = [-1, -3, 0]


### 3.3. "Lát cắt" và luồng bắt buộc qua ranh giới

Cắt hàng máy tại ranh giới giữa máy `i` và `i+1`. Bên trái thừa `P[i]` bộ (nếu âm là thiếu). Muốn cân bằng, **đúng** `|P[i]|` bộ phải đi qua ranh giới đó (theo chiều từ bên thừa sang bên thiếu) — không cách nào tránh.

Mỗi lượt, qua một ranh giới, **lượng ròng** đi qua tối đa là 1 (máy `i` cho sang phải 1 bộ; nếu đồng thời máy `i+1` cho sang trái 1 bộ thì ròng là 0). Vậy cần **ít nhất `|P[i]|` lượt**.

In [4]:
for i in range(len(machines) - 1):
    print(f"ranh giới {i}|{i+1}: bên trái thừa {P[i]:+d} -> cần ít nhất {abs(P[i])} lượt")

ranh giới 0|1: bên trái thừa -1 -> cần ít nhất 1 lượt
ranh giới 1|2: bên trái thừa -3 -> cần ít nhất 3 lượt


### 3.4. BFS trên trạng thái và `itertools.product`

Để kiểm chứng, ta sẽ BFS trên trạng thái `tuple(machines)`. Mỗi lượt, mỗi máy chọn một trong 3 hành động: đứng yên (`0`), cho trái (`-1`), cho phải (`+1`). `itertools.product([0, -1, 1], repeat=n)` sinh mọi tổ hợp `3^n` hành động.

In [5]:
from itertools import product
acts = list(product([0, -1, 1], repeat=2))
print(len(acts), acts)

9 [(0, 0), (0, -1), (0, 1), (-1, 0), (-1, -1), (-1, 1), (1, 0), (1, -1), (1, 1)]


### 3.5. `max` trên nhiều đại lượng

Đáp án sẽ là `max` của nhiều chặn dưới. Python: `max(a, b, c)` hoặc `max(iterable, default=0)`.

In [6]:
print(max(3, 1, 2), max([abs(x) for x in [-4, 2, 1]]), max([], default=0))

3 4 0


## 4. Ý tưởng nền tảng / trực giác

Thử tính tay `[0, 3, 0]`, `avg = 1`, `d = [-1, 2, -1]`, `P = [-1, 1, 0]`:
- Ranh giới 0|1: bên trái thiếu 1 → cần ≥ 1 lượt.
- Ranh giới 1|2: bên trái thừa 1 → cần ≥ 1 lượt.
- Nhưng đáp án là 2! Vì máy giữa phải **cho đi 2 bộ** (thừa 2), và mỗi lượt một máy chỉ cho đi 1 bộ → ≥ 2 lượt.

Vậy có **hai loại nút thắt**:
1. **Ranh giới (cạnh):** cần ≥ `|P[i]|` lượt.
2. **Máy (đỉnh) thừa:** máy thừa `d[i] > 0` phải cho đi `d[i]` bộ → cần ≥ `d[i]` lượt.

Vì sao chỉ máy **thừa** mà không phải máy **thiếu**? Một máy có thể **nhận đồng thời** từ hai hàng xóm (2 bộ/lượt), nên máy thiếu không tạo nút thắt riêng — nút thắt của nó đã được tính qua ranh giới.

Thử `[1, 0, 5]`: `avg = 2`, `d = [-1, -2, 3]`, `P = [-1, -3, 0]`. Ranh giới cần `max(1, 3) = 3`; máy thừa nhiều nhất `3`. Đáp án 3. ✔

In [7]:
def bounds(machines):
    n = len(machines); avg = sum(machines) // n
    d = [x - avg for x in machines]
    P = list(accumulate(d))
    edge = max((abs(p) for p in P[:-1]), default=0)
    node = max(d)
    return d, P, edge, node

for arr in [[0, 3, 0], [1, 0, 5], [4, 0, 0, 4]]:
    d, P, e, v = bounds(arr)
    print(f"{arr}: d={d} P={P} chặn cạnh={e} chặn máy={v} -> max={max(e, v)}")

[0, 3, 0]: d=[-1, 2, -1] P=[-1, 1, 0] chặn cạnh=1 chặn máy=2 -> max=2
[1, 0, 5]: d=[-1, -2, 3] P=[-1, -3, 0] chặn cạnh=3 chặn máy=3 -> max=3
[4, 0, 0, 4]: d=[2, -2, -2, 2] P=[2, 0, -2, 0] chặn cạnh=2 chặn máy=2 -> max=2


## 5. Cách 1 — Brute Force: BFS trên mọi trạng thái

**Ý tưởng.** Trạng thái = bộ đếm của các máy. Từ một trạng thái, thử mọi tổ hợp hành động (`3^n`), loại tổ hợp không hợp lệ (máy rỗng mà cho, hoặc cho ra ngoài hàng), sinh trạng thái mới. BFS cho số lượt ít nhất tới trạng thái `(avg, avg, …, avg)`.

Lưu ý: máy chỉ cần có ≥ 1 bộ **ở đầu lượt** để cho đi (nhận và cho diễn ra đồng thời).

In [8]:
from collections import deque

def min_moves_bfs(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    target = tuple([s // n] * n)
    start = tuple(machines)
    if start == target:
        return 0
    dist = {start: 0}
    q = deque([start])
    while q:
        cur = q.popleft()
        for act in product((0, -1, 1), repeat=n):
            nxt = list(cur)
            ok = True
            for i, a in enumerate(act):
                if a == 0:
                    continue
                j = i + a
                if cur[i] == 0 or not (0 <= j < n):
                    ok = False; break
                nxt[i] -= 1; nxt[j] += 1
            if not ok:
                continue
            t = tuple(nxt)
            if t not in dist:
                dist[t] = dist[cur] + 1
                if t == target:
                    return dist[t]
                q.append(t)
    return -1

print(min_moves_bfs([1, 0, 5]), min_moves_bfs([0, 3, 0]), min_moves_bfs([0, 2, 0]))
assert min_moves_bfs([1, 0, 5]) == 3 and min_moves_bfs([0, 3, 0]) == 2 and min_moves_bfs([0, 2, 0]) == -1

3 2 -1


**Độ phức tạp:** số trạng thái = số cách chia `tổng` bộ vào `n` máy = `C(tổng + n - 1, n - 1)`, mỗi trạng thái thử `3^n` tổ hợp. Với `n = 10^4`, `tổng = 10^9` thì không tưởng.

**Nút thắt:** vét cạn hoàn toàn không dùng cấu trúc của bài. Ta cần một công thức dựa trên **chặn dưới** ở mục 4 — và chứng minh chặn dưới **đạt được**.

In [9]:
import time
t = time.perf_counter()
print(min_moves_bfs([6, 0, 0, 0, 0, 6]), f"({time.perf_counter() - t:.2f}s cho 6 máy, 12 bộ)")

4 (0.03s cho 6 máy, 12 bộ)


## 6. Bước đột phá

**Insight 1 — Hai chặn dưới.**
- (Cạnh) Với mỗi ranh giới `i | i+1`: số lượt `>= |P[i]|`.
- (Máy) Với mỗi máy `i` thừa: số lượt `>= d[i]`.

**Insight 2 — Chặn dưới đạt được.** Đáp án chính xác là

```text
ans = max( max_i |P[i]| , max_i d[i] )
```

(Đây là kết quả đã biết của bài; chứng minh chiều "đạt được" khá dài — ý chính: mỗi lượt có thể cho **mọi** máy đang phải gửi đồ theo hướng cần thiết cùng gửi một bộ, làm giảm **đồng thời** mọi chặn dưới đang lớn nhất đi 1. Notebook kiểm chứng bằng BFS vét cạn ở mục 12.)

**Insight 3 — Góc nhìn theo từng máy (luồng trái/phải).** Với máy `i`, gọi:
- `L = P[i-1]` = bên trái máy `i` thừa bao nhiêu (âm: thiếu → máy `i` phải gửi sang trái `-L` bộ).
- `R` = tổng các `d` **bên phải** máy `i` = bên phải thừa bao nhiêu (vì tổng mọi `d` bằng 0 nên `R = −P[i]`).

Nếu **cả hai bên đều thiếu** (`L < 0` và `R < 0`) thì máy `i` phải **gửi cả hai phía**, mỗi lượt chỉ gửi 1 bộ → cần `-L - R` lượt. Ngược lại, cần `max(|L|, |R|)` lượt. Đáp án = max qua mọi máy. Đây là Cách 3; hai công thức cho cùng kết quả.

**Quy tắc:** **nếu** `tổng % n != 0` **thì** `-1`; ngược lại tính `max` của các chặn dưới.

In [10]:
# Minh họa Insight 3 với [0, 3, 0]: máy giữa phải gửi cả hai phía
arr = [0, 3, 0]; avg = 1
for i in range(len(arr)):
    L = sum(arr[:i]) - i * avg
    R = sum(arr[i + 1:]) - (len(arr) - i - 1) * avg
    need = -L - R if (L < 0 and R < 0) else max(abs(L), abs(R))
    print(f"máy {i}: trái thừa {L:+d}, phải thừa {R:+d} -> cần {need}")

máy 0: trái thừa +0, phải thừa +1 -> cần 1
máy 1: trái thừa -1, phải thừa -1 -> cần 2
máy 2: trái thừa +1, phải thừa +0 -> cần 1


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Công thức tổng tiền tố (một lượt duyệt)

Duyệt một lần, giữ `run` = tổng tiền tố của `d`. Cập nhật `ans = max(ans, abs(run), d_i)`.

Chú ý: dùng `abs(run)` cho cạnh, nhưng chỉ `d_i` (không lấy trị tuyệt đối) cho máy — máy thiếu không phải nút thắt.

In [11]:
def min_moves_prefix(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n
    ans = run = 0
    for x in machines:
        d = x - avg
        run += d
        ans = max(ans, abs(run), d)
    return ans

print(min_moves_prefix([1, 0, 5]), min_moves_prefix([0, 3, 0]), min_moves_prefix([0, 2, 0]))

3 2 -1


**Code nộp LeetCode (Cách 2):** xem mục 9.

**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(1)`.

### 7.2. Cách 3 — Luồng trái/phải tại từng máy

Tính trước tổng tiền tố `pre[i] = machines[0] + … + machines[i-1]`. Với máy `i`:
- `L = pre[i] - i·avg`,
- `R = (s - pre[i] - machines[i]) - (n - i - 1)·avg`.
- `need_i = -L - R` nếu `L < 0 and R < 0`, ngược lại `max(|L|, |R|)`.

Đáp án `max(need_i)`.

In [12]:
def min_moves_per_machine(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n
    ans = 0
    pre = 0
    for i, x in enumerate(machines):
        L = pre - i * avg
        R = (s - pre - x) - (n - i - 1) * avg
        if L < 0 and R < 0:
            need = -L - R
        else:
            need = max(abs(L), abs(R))
        ans = max(ans, need)
        pre += x
    return ans

print(min_moves_per_machine([1, 0, 5]), min_moves_per_machine([0, 3, 0]), min_moves_per_machine([0, 2, 0]))

3 2 -1


**Code nộp LeetCode (Cách 3):**

In [13]:
from typing import List

class SolutionPerMachine:
    def findMinMoves(self, machines: List[int]) -> int:
        return min_moves_per_machine(machines)

print(SolutionPerMachine().findMinMoves([1, 0, 5]))

3


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)`.

**Vì sao Cách 2 = Cách 3?** Với máy `i`: `L = P[i-1]`, `R = −P[i]` (vì tổng toàn bộ bằng 0). `d[i] = P[i] - P[i-1] = −R − L`.
- Nếu `L < 0, R < 0`: `need_i = −L − R = d[i] > 0` — chính là chặn máy.
- Ngược lại `need_i = max(|L|, |R|) = max(|P[i-1]|, |P[i]|)` — chặn cạnh hai bên máy `i`; và trong trường hợp này `d[i] <= max(|L|, |R|)` nên không mất gì.
Lấy max trên mọi `i` ra cùng một đại lượng.

### 7.3. (Mở rộng) Cách 2b — Mô phỏng tham lam có kiểm chứng

Để "thấy" chặn dưới đạt được, ta viết một mô phỏng: mỗi lượt, mỗi máy quyết định gửi theo hướng có **luồng bắt buộc** còn lại lớn nhất (ưu tiên cạnh cần gửi nhiều hơn), miễn máy có hàng. Mô phỏng này **không** phải lời giải chính thức (không chứng minh tổng quát ở đây), nhưng ta đo số lượt của nó và so với công thức trên nhiều input ngẫu nhiên.

In [14]:
def simulate_greedy(machines, limit=1000):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n
    m = list(machines)
    # flow[i] = số bộ còn phải đi từ i sang i+1 (âm: từ i+1 sang i)
    flow = list(accumulate(x - avg for x in m))[:-1]
    moves = 0
    while any(flow):
        moves += 1
        if moves > limit:
            return None
        give = [0] * n                     # -1: cho trái, +1: cho phải
        for i in range(n):
            right_need = flow[i] if i < n - 1 and flow[i] > 0 else 0
            left_need = -flow[i - 1] if i > 0 and flow[i - 1] < 0 else 0
            if m[i] == 0 or (right_need == 0 and left_need == 0):
                continue
            give[i] = 1 if right_need >= left_need else -1
        for i in range(n):
            if give[i] == 1:
                m[i] -= 1; m[i + 1] += 1; flow[i] -= 1
            elif give[i] == -1:
                m[i] -= 1; m[i - 1] += 1; flow[i - 1] += 1
    return moves

print(simulate_greedy([1, 0, 5]), simulate_greedy([0, 3, 0]), simulate_greedy([4, 0, 0, 4]))

3 2 2


Mô phỏng tham lam này cho kết quả bằng công thức trên các ví dụ trên; ở mục 12 ta sẽ so trên nhiều input ngẫu nhiên và **ghi nhận** số trường hợp chênh lệch (trong thử nghiệm của notebook là 0) (một heuristic mô phỏng không nhất thiết luôn tối ưu — đó là lý do ta dùng công thức đã chứng minh làm lời giải nộp).

## 8. TRACE từng bước

Trace **Cách 2** trên `[1, 0, 5]` (`avg = 2`):

| i | x | d = x − avg | run (tổng tiền tố) | abs(run) | ans |
|---|---|---|---|---|---|
| 0 | 1 | −1 | −1 | 1 | 1 |
| 1 | 0 | −2 | −3 | 3 | 3 |
| 2 | 5 | 3 | 0 | 0 | 3 |

In [15]:
def trace_prefix(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        print("tổng", s, "không chia hết cho", n, "-> -1")
        return -1
    avg = s // n
    ans = run = 0
    print(f"avg = {avg}")
    for i, x in enumerate(machines):
        d = x - avg
        run += d
        old = ans
        ans = max(ans, abs(run), d)
        why = "" if ans == old else (" <- chặn cạnh" if ans == abs(run) else " <- chặn máy")
        print(f"i={i}: x={x:>2} d={d:+d} run={run:+d} |run|={abs(run)} ans={ans}{why}")
    return ans

trace_prefix([1, 0, 5])
print()
trace_prefix([0, 3, 0])

avg = 2
i=0: x= 1 d=-1 run=-1 |run|=1 ans=1 <- chặn cạnh
i=1: x= 0 d=-2 run=-3 |run|=3 ans=3 <- chặn cạnh
i=2: x= 5 d=+3 run=+0 |run|=0 ans=3

avg = 1
i=0: x= 0 d=-1 run=-1 |run|=1 ans=1 <- chặn cạnh
i=1: x= 3 d=+2 run=+1 |run|=1 ans=2 <- chặn máy
i=2: x= 0 d=-1 run=+0 |run|=0 ans=2


2

Trace **mô phỏng** để thấy từng lượt trên `[1, 0, 5]` và `[0, 0, 11, 5]`:

In [16]:
def trace_simulation(machines):
    n, s = len(machines), sum(machines); avg = s // n
    m = list(machines)
    flow = list(accumulate(x - avg for x in m))[:-1]
    t = 0
    print(f"lượt {t}: {m}  luồng còn lại={flow}")
    while any(flow) and t < 100:
        t += 1
        give = [0] * n
        for i in range(n):
            rn = flow[i] if i < n - 1 and flow[i] > 0 else 0
            ln = -flow[i - 1] if i > 0 and flow[i - 1] < 0 else 0
            if m[i] and (rn or ln):
                give[i] = 1 if rn >= ln else -1
        for i in range(n):
            if give[i] == 1: m[i] -= 1; m[i + 1] += 1; flow[i] -= 1
            elif give[i] == -1: m[i] -= 1; m[i - 1] += 1; flow[i - 1] += 1
        arrows = "".join({0: ".", 1: ">", -1: "<"}[g] for g in give)
        print(f"lượt {t}: {m}  hành động={arrows}  luồng còn lại={flow}")
    print("công thức cho:", min_moves_prefix(machines))

trace_simulation([1, 0, 5])
print()
trace_simulation([0, 0, 11, 5])

lượt 0: [1, 0, 5]  luồng còn lại=[-1, -3]
lượt 1: [1, 1, 4]  hành động=..<  luồng còn lại=[-1, -2]
lượt 2: [2, 1, 3]  hành động=.<<  luồng còn lại=[0, -1]
lượt 3: [2, 2, 2]  hành động=..<  luồng còn lại=[0, 0]
công thức cho: 3

lượt 0: [0, 0, 11, 5]  luồng còn lại=[-4, -8, -1]
lượt 1: [0, 1, 11, 4]  hành động=..<<  luồng còn lại=[-4, -7, 0]
lượt 2: [1, 1, 10, 4]  hành động=.<<.  luồng còn lại=[-3, -6, 0]
lượt 3: [2, 1, 9, 4]  hành động=.<<.  luồng còn lại=[-2, -5, 0]
lượt 4: [3, 1, 8, 4]  hành động=.<<.  luồng còn lại=[-1, -4, 0]
lượt 5: [4, 1, 7, 4]  hành động=.<<.  luồng còn lại=[0, -3, 0]
lượt 6: [4, 2, 6, 4]  hành động=..<.  luồng còn lại=[0, -2, 0]
lượt 7: [4, 3, 5, 4]  hành động=..<.  luồng còn lại=[0, -1, 0]
lượt 8: [4, 4, 4, 4]  hành động=..<.  luồng còn lại=[0, 0, 0]
công thức cho: 8


## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def findMinMoves(self, machines: List[int]) -> int:
        n, total = len(machines), sum(machines)
        if total % n:
            return -1
        avg = total // n
        ans = run = 0
        for x in machines:
            d = x - avg              # máy này thừa (+) / thiếu (-)
            run += d                 # lượng phải đi qua ranh giới bên phải máy này
            ans = max(ans, abs(run), d)
        return ans

sol = Solution()
assert sol.findMinMoves([1, 0, 5]) == 3
assert sol.findMinMoves([0, 3, 0]) == 2
assert sol.findMinMoves([0, 2, 0]) == -1
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Điều kiện cần (−1).** Tổng không đổi qua các lượt; trạng thái đích có tổng `n · avg` → tổng phải chia hết cho `n`.

**Chặn dưới 1 (cạnh).** Xét ranh giới giữa máy `i` và `i+1`. Tổng bên trái phải thay đổi từ `pre` thành `(i+1)·avg`, tức đổi `−P[i]`. Mỗi lượt, tổng bên trái chỉ đổi do máy `i` gửi sang phải (−1) và/hoặc máy `i+1` gửi sang trái (+1) → đổi tối đa 1 về trị tuyệt đối. Vậy cần ≥ `|P[i]|` lượt.

**Chặn dưới 2 (máy).** Máy `i` phải giảm từ `machines[i]` xuống `avg` một lượng ròng `d[i]`. Mỗi lượt nó gửi đi tối đa 1 bộ (nhận vào chỉ làm tăng). Vậy cần ≥ `d[i]` lượt. (Máy thiếu nhận được tối đa 2 bộ/lượt và đã bị chặn bởi các cạnh hai bên, nên không cho thêm chặn mạnh hơn.)

**Đạt được (chặn trên).** Kết quả đã biết: luôn tồn tại cách xếp lịch sao cho mỗi lượt, mọi chặn dưới đang đạt giá trị lớn nhất đều giảm 1 — do đó số lượt bằng đúng `max` của các chặn dưới. Chứng minh đầy đủ cần lập luận về luồng trên đường thẳng; ở đây ta **kiểm chứng bằng BFS vét cạn** trên hàng nghìn input nhỏ ngẫu nhiên (mục 12) — BFS cho đáp án tối ưu tuyệt đối, và công thức khớp hoàn toàn.

**Cách 2 = Cách 3:** đã chỉ ra ở mục 7.2.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. BFS | `O(C(S+n-1, n-1) · 3^n · n)` | số trạng thái |
| 2. Tiền tố | `O(n)` | `O(1)` |
| 3. Trái/phải từng máy | `O(n)` | `O(1)` |
| (Mô phỏng) | `O(ans · n)` | `O(n)` |

In [18]:
import random
random.seed(517)
big = [random.randint(0, 10**5) for _ in range(10**4)]
big[-1] += (-sum(big)) % len(big)       # chỉnh để tổng chia hết
for name, f in [("prefix", min_moves_prefix), ("per_machine", min_moves_per_machine), ("Solution", Solution().findMinMoves)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:<12}: {time.perf_counter() - t:.4f}s -> {r}")

prefix      : 0.0007s -> 2436377
per_machine : 0.0020s -> 2436377
Solution    : 0.0009s -> 2436377


## 12. Bộ test

In [19]:
methods = {
    "bfs": min_moves_bfs,
    "prefix": min_moves_prefix,
    "per_machine": min_moves_per_machine,
    "SolPerMachine": SolutionPerMachine().findMinMoves,
    "Solution": Solution().findMinMoves,
}
cases = [
    ([1, 0, 5], 3),
    ([0, 3, 0], 2),
    ([0, 2, 0], -1),
    ([5], 0),                 # một máy
    ([0], 0),
    ([2, 2, 2], 0),           # đã cân bằng
    ([4, 0, 0, 4], 2),        # hai đầu cùng gửi vào giữa
    ([0, 0, 11, 5], 8),       # máy 2 thừa 7, cạnh 1|2 cần 8
    ([1, 2], -1),
]
for arr, exp in cases:
    for name, f in methods.items():
        if name == "bfs" and sum(arr) > 12:
            continue
        got = f(list(arr))
        assert got == exp, (name, arr, got, exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [20]:
random.seed(2026)
checked = 0
for trial in range(1500):
    n = random.randint(1, 5)
    arr = [random.randint(0, 4) for _ in range(n)]
    if sum(arr) > 10:
        continue
    exp = min_moves_bfs(arr)
    for name, f in methods.items():
        got = f(list(arr))
        assert got == exp, (name, arr, got, exp)
    checked += 1
print(f"{checked} input ngẫu nhiên (seed 2026): công thức khớp BFS vét cạn")

1312 input ngẫu nhiên (seed 2026): công thức khớp BFS vét cạn


In [21]:
# So sánh mô phỏng tham lam với công thức trên input lớn hơn (ghi nhận, không assert)
random.seed(7)
diff = 0; total_cases = 0
for _ in range(2000):
    n = random.randint(1, 8)
    arr = [random.randint(0, 9) for _ in range(n)]
    if sum(arr) % n:
        continue
    total_cases += 1
    if simulate_greedy(arr) != min_moves_prefix(arr):
        diff += 1
print(f"mô phỏng tham lam khác công thức ở {diff}/{total_cases} trường hợp")

mô phỏng tham lam khác công thức ở 0/672 trường hợp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Lấy trị tuyệt đối cho cả chặn máy

**Code sai:** `ans = max(ans, abs(run), abs(d))`.

**Vì sao sai:** máy **thiếu** có thể nhận 2 bộ/lượt từ hai phía, nên `|d|` của máy thiếu không phải chặn dưới. Ví dụ `[3, 0, 3]`: `avg = 2`, máy giữa thiếu 2 nhưng nhận đủ từ hai bên chỉ trong **1** lượt.

In [22]:
def wrong_abs(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n; ans = run = 0
    for x in machines:
        d = x - avg; run += d
        ans = max(ans, abs(run), abs(d))     # SAI
    return ans

print("sai:", wrong_abs([3, 0, 3]), "| đúng:", Solution().findMinMoves([3, 0, 3]), "| BFS:", min_moves_bfs([3, 0, 3]))

sai: 2 | đúng: 1 | BFS: 1


**Code đúng:** `max(ans, abs(run), d)`.

### Lỗi 2 — Chỉ dùng chặn cạnh (quên chặn máy)

**Code sai:** `ans = max(ans, abs(run))`.

In [23]:
def wrong_edge_only(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n; ans = run = 0
    for x in machines:
        run += x - avg
        ans = max(ans, abs(run))
    return ans

print("sai:", wrong_edge_only([0, 3, 0]), "| đúng:", Solution().findMinMoves([0, 3, 0]))

sai: 1 | đúng: 2


**Vì sao sai:** máy giữa của `[0, 3, 0]` phải gửi 2 bộ (mỗi lượt 1), nhưng mỗi cạnh chỉ cần 1.

### Lỗi 3 — Cộng tổng các độ lệch dương (tư duy "mỗi lượt chuyển 1 bộ")

**Code sai:** `sum(max(0, x - avg) for x in machines)` hoặc `sum(abs(run))`.

**Vì sao sai:** đề cho **nhiều máy chuyển đồng thời** trong một lượt. Tổng số bộ phải di chuyển (hay tổng `|P[i]|`) là số **bước chuyển** khi mỗi lượt chỉ 1 bộ đi 1 nấc — một bài khác (giống LC 979 Distribute Coins in Binary Tree trên đường thẳng).

In [24]:
arr = [1, 0, 5]
avg = sum(arr) // len(arr)
print("tổng |P[i]| =", sum(abs(p) for p in accumulate(x - avg for x in arr)), "(số bước nếu mỗi lượt chỉ 1 bộ đi 1 nấc)")
print("đáp án đúng:", Solution().findMinMoves(arr))

tổng |P[i]| = 4 (số bước nếu mỗi lượt chỉ 1 bộ đi 1 nấc)
đáp án đúng: 3


### Lỗi 4 — Quên kiểm tra chia hết hoặc dùng `/`

`avg = total / n` ra số thực, `x - avg` thành float; nếu không chia hết mà vẫn chạy sẽ ra số vô nghĩa. Luôn kiểm tra `total % n` trước và dùng `//`.

## 14. Mẹo & nhận dạng pattern

- "Cân bằng giá trị trên một **đường thẳng**, chuyển giữa các phần tử kề" → **tổng tiền tố = luồng qua ranh giới**.
- Hỏi "số **lượt** ít nhất khi nhiều thao tác diễn ra **song song**" → đáp án thường là **max** của các chặn dưới (nút thắt), không phải tổng.
- Tìm chặn dưới từ **cạnh** (luồng qua lát cắt) và từ **đỉnh** (năng lực gửi/nhận mỗi lượt).
- Kiểm chứng công thức "đẹp" bằng BFS vét cạn trên input nhỏ — đặc biệt khi chứng minh chiều "đạt được" khó.
- Họ hàng: LC 979 Distribute Coins in Binary Tree (tổng luồng), LC 2141, bài toán "min cost to equalize array" bằng prefix.

## 15. Biến thể

1. **Mỗi lượt chỉ một bộ đi một nấc** → đáp án là `Σ |P[i]|` (tổng luồng).
2. **Máy xếp thành vòng tròn** → luồng có một ẩn số "luồng vòng" `c`; chọn `c` để tối thiểu `max` → tìm kiếm/median.
3. **Mỗi máy gửi tối đa `k` bộ/lượt** → chặn máy thành `ceil(d[i] / k)`, chặn cạnh vẫn cần xem xét lại.
4. **Cây thay vì đường thẳng** (LC 979) → luồng qua mỗi cạnh cây bằng tổng độ lệch của cây con.

Demo biến thể 1:

In [25]:
def total_single_moves(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n
    return sum(abs(p) for p in accumulate(x - avg for x in machines))

print(total_single_moves([1, 0, 5]), total_single_moves([0, 3, 0]))

4 2


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | Vét cạn trạng thái, `3^n` hành động mỗi lượt | mũ | mũ | Kiểm chứng `n`, tổng nhỏ |
| 2. Tiền tố | `max(|P[i]|, d[i])` | `O(n)` | `O(1)` | Lời giải nộp |
| 3. Trái/phải từng máy | Hai bên cùng thiếu → cộng; ngược lại → max | `O(n)` | `O(1)` | Trực giác "theo máy", dễ giải thích |
| (Mô phỏng) | Gửi theo hướng luồng còn lớn | `O(ans·n)` | `O(n)` | Minh họa lịch chuyển cụ thể |

## 17. Bài tập

**Bài 1 (đoán output).** `Solution().findMinMoves([0, 0, 6])` = ?

**Bài 2 (tính tay).** Tính `d`, `P` và đáp án cho `[2, 4, 0, 6]`.

**Bài 3 (điền TODO).** Hoàn thiện vòng lặp của Cách 2.

In [26]:
def min_moves_todo(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n
    ans = run = 0
    for x in machines:
        # TODO: d = x - avg; run += d; ans = max(ans, abs(run), d)
        pass
    return ans

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert min_moves_todo([1, 0, 5]) == 3

**Bài 4 (tự viết lại).** Viết lại Cách 3 không nhìn code.

**Bài 5 (mini challenge).** Biến thể "mỗi máy gửi tối đa `k` bộ mỗi lượt (có thể chia hai phía), qua mỗi ranh giới ròng tối đa `k` bộ mỗi lượt". Đề xuất công thức `max(ceil(|P[i]|/k), ceil(d[i]/k))` và **kiểm chứng bằng BFS** với `k = 2` trên input nhỏ. Nếu không khớp, hãy tìm phản ví dụ.

In [27]:
def min_moves_k_todo(machines, k):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: `avg = 2`.
- Mức 2: Bài 1: `d = [-2, -2, 4]`, `P = [-2, -4, 0]`.
- Mức 3: Bài 1: `max(4, 4) = 4`. Bài 2: `avg = 3`, `d = [-1, 1, -3, 3]`, `P = [-1, 0, -3, 0]` → 3.

**Bài 3**
- Mức 1: Ba dòng.
- Mức 2: `abs` chỉ cho `run`.
- Mức 3: Trả `ans`.

**Bài 4**
- Mức 1: Giữ `pre` = tổng các máy bên trái.
- Mức 2: `L = pre - i*avg`, `R = (s - pre - x) - (n-i-1)*avg`.
- Mức 3: Hai bên cùng âm thì `-L-R`.

**Bài 5**
- Mức 1: BFS: mỗi máy chọn số bộ gửi trái `a` và phải `b` với `a + b <= min(k, m[i])`.
- Mức 2: Ràng buộc cạnh tự động thỏa vì chỉ hai máy kề tác động.
- Mức 3: So công thức với BFS; kết quả kiểm chứng ở đáp án.

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
assert Solution().findMinMoves([0, 0, 6]) == 4 == min_moves_bfs([0, 0, 6])
assert Solution().findMinMoves([2, 4, 0, 6]) == 3

# Bài 3
def min_moves_sol(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n; ans = run = 0
    for x in machines:
        d = x - avg; run += d; ans = max(ans, abs(run), d)
    return ans
assert min_moves_sol([1, 0, 5]) == 3

# Bài 4
def per_machine_ex(machines):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg, pre, ans = s // n, 0, 0
    for i, x in enumerate(machines):
        L = pre - i * avg
        R = s - pre - x - (n - i - 1) * avg
        ans = max(ans, -L - R if L < 0 and R < 0 else max(abs(L), abs(R)))
        pre += x
    return ans

random.seed(31)
for _ in range(500):
    arr = [random.randint(0, 20) for _ in range(random.randint(1, 10))]
    assert per_machine_ex(arr) == min_moves_sol(arr) == Solution().findMinMoves(arr)
print("Bài 1–4 OK")

Bài 1–4 OK


In [29]:
# Bài 5: BFS cho biến thể k bộ/lượt và so với công thức đề xuất
def bfs_k(machines, k):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    target = tuple([s // n] * n); start = tuple(machines)
    if start == target:
        return 0
    # mỗi máy chọn (gửi trái a, gửi phải b), a + b <= min(k, số bộ)
    def options(i, cnt):
        res = []
        for a in range(0, min(k, cnt) + 1):
            for b in range(0, min(k, cnt) - a + 1):
                if (a and i == 0) or (b and i == n - 1):
                    continue
                res.append((a, b))
        return res
    dist = {start: 0}; q = deque([start])
    while q:
        cur = q.popleft()
        for combo in product(*[options(i, cur[i]) for i in range(n)]):
            nxt = list(cur)
            for i, (a, b) in enumerate(combo):
                nxt[i] -= a + b
                if a: nxt[i - 1] += a
                if b: nxt[i + 1] += b
            t = tuple(nxt)
            if t not in dist:
                dist[t] = dist[cur] + 1
                if t == target:
                    return dist[t]
                q.append(t)
    return -1

def formula_k(machines, k):
    n, s = len(machines), sum(machines)
    if s % n:
        return -1
    avg = s // n; ans = run = 0
    for x in machines:
        d = x - avg; run += d
        ans = max(ans, -(-abs(run) // k), -(-d // k) if d > 0 else 0)
    return ans

random.seed(41)
mismatch = []
for _ in range(400):
    n = random.randint(1, 4)
    arr = [random.randint(0, 5) for _ in range(n)]
    if sum(arr) % n or sum(arr) > 10:
        continue
    b, f = bfs_k(arr, 2), formula_k(arr, 2)
    if b != f:
        mismatch.append((arr, b, f))
print("số input lệch giữa công thức đề xuất và BFS (k=2):", len(mismatch), "| ví dụ:", mismatch[:3])
assert formula_k([1, 0, 5], 1) == 3 and bfs_k([1, 0, 5], 1) == 3

số input lệch giữa công thức đề xuất và BFS (k=2): 1 | ví dụ: [([3, 1, 0, 0], 2, 1)]


**Nhận xét Bài 5:** BFS tìm ra phản ví dụ `[3, 1, 0, 0]` với `k = 2`: công thức "chia `k`" cho 1 lượt, nhưng thật ra cần 2. Lý do: `avg = 1`, máy 0 thừa 2 bộ, và **một bộ phải đi tới máy 3 — cách 3 nấc**. Trong một lượt, mỗi bộ đồ chỉ đi được **1 nấc**, nên khoảng cách cũng là một chặn dưới! Với luật gốc (`k = 1`), chặn này bị "che" bởi chặn cạnh/máy, nhưng khi `k` lớn thì nó lộ ra. Bài học: **không** tự động tổng quát hóa công thức khi đổi luật; luôn kiểm chứng bằng vét cạn.

In [30]:
assert bfs_k([3, 1, 0, 0], 2) == 2 and formula_k([3, 1, 0, 0], 2) == 1
print("Phản ví dụ được xác nhận: [3,1,0,0], k=2 -> BFS 2, công thức 1")

Phản ví dụ được xác nhận: [3,1,0,0], k=2 -> BFS 2, công thức 1


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
n, s = len(a), sum(a)
if s % n: return -1
avg, ans, run = s // n, 0, 0
for x in a:
    d = x - avg
    run += d                      # luồng qua ranh giới bên phải
    ans = max(ans, abs(run), d)   # chặn cạnh |run|, chặn máy d (chỉ dương)
return ans
```

**Checklist trước khi submit:**

- [ ] Kiểm tra `sum % n` → `-1`.
- [ ] Dùng `//` cho trung bình.
- [ ] Chặn cạnh dùng `abs(run)`; chặn máy dùng `d` **không** lấy trị tuyệt đối.
- [ ] Trả về `max`, không phải tổng.
- [ ] Test `[0,3,0]` (=2), `[3,0,3]` (=1), một máy, đã cân bằng.
- [ ] Đối chiếu với BFS vét cạn trên input nhỏ.